持久化与记忆
- 基本运用：线程隔离的持久化
- 基本运用：跨线程持久化调用
- 记忆：短期记忆及实现
- 记忆：长期记忆及实现
- 记忆：使用总结技术优化记忆

线程隔离的持久化层
***

In [ ]:
from langchain_deepseek import ChatDeepSeek
import os
from langgraph.graph import StateGraph, MessagesState, START

model = ChatDeepSeek()

def call_model(state: MessagesState):
   response = model.invoke(state['messages'])
   return { "messages": response }

builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
graph = builder.compile()

激活持久化层

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 使用MemorySaver保持中间状态
saver = MemorySaver()
graph = builder.compiler(checkpointer=memory)

In [ ]:
config = {"configurable": {"thread_id": "1"}}
input_messages = {"role": "user", "content": "hi, 我是tomie"}
for chunk in graph.stream({"messages": [input_messages]}, config, stream_model="values"):
    chunk["messages"][-1].pretty_print()


跨线程共享持久化数据
***
- userid

- 设置内在记忆

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings
import os

# 使用内存存储来保存向量化后的记忆数据
in_memory_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(
            model="Pro/BAAI/bge-m3",
            api_key=os.environ.get('DEEPSEEK_API_KEY'),
            base_url="...",
        ),
        "dims": 1024,
    }
)

In [ ]:
import uuid
from typing import Annotated
from typing_extensions import TypedDict
from langchain_deepseek import ChatDeepSeek
import os 
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessageState, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore

model = ChatDeepSeek()

def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    # 从存储中检索用户信息：
    user_id = config['configurable']['user_id']
    # 从存储中检索用户信息：
    namespace = ('memories', user_id)
    memories = store.search(namespace, query=str(state['messages'][-1].content)) # 消息的最后一条作为查询
    info = "\n".join([d.value['data'] for d in memories])
    system_msg = f"你是一个正在与用户交谈的小助手。用户信息：{info}"

    # 如果用户要求模型记住信息，则存储新的记忆
    last_message = state['messages'][-1] # 最后一条消息
    if "记住" in last_message.content.lower() or "remember" in last_message.content.lower():
        # 硬编码一个记忆
        memory = "用户的名字是tomiezhang"
        store.put(namespace, str(uuid.uuid4()), {"data": memory})

    response = model.invoke(
       [{ "role": "system", "content": system_msg }] + state['messages']
    )
    return {"messages": response}

builder = StateGraph(MessageState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

## 注意：我们在编译图时传递了store对象
graph = builder.compile(checkpointer=MemorySaver(), store=in_memory_store)

In [ ]:
config = {"configurable": {"thread_id": "1", "user_id": "1"}}
input_message = {"rule": "user", "content": "请记住我的名字叫tomiezhang"}
for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk['messages'][-1].pretty_print()

In [ ]:
config = {"configurable": {"thread_id": "2", "user_id": "1"}}
input_message = {"rule": "user", "content": "我叫什么名字"}
for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk['messages'][-1].pretty_print()

短期记忆的实现
***
- 基于最简单的ReAct智能体

In [ ]:
from typing import Literal
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool
from langgraph.checkpointer.memory import MemorySaver
from langgraph.graph import MessageState, StateGraph, START, END
from langgraph.prebuilt import ToolNode

# 注意：使用内存存储记忆
memory = MemorySaver()

@tool
def search(query: str):
    """调用此函数可以浏览网络"""
    # 模拟一个网络搜索返回
    return "北京天气晴朗，大约22度，温度30%"

tools = [search]
tool_node = ToolNode(tools)
model = ChatDeepSeek(
    model="Pro/deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE")
)
bound_model = model.bind_tools(tools)

def should_continue(state: MessagesState):
    """返回下一个要执行的节点"""
    last_message = state["messages"][-1]
    # 如果没有函数调用，则结束
    if not last_message.tool_calls:
        return END
    # 否则如果有，我们继续
    return "action"

# 定义调用模型函数
def call_model(state: MessagesState):
    response = bound_model.invoke(state["messages"])
    # 返回一个列表，因为这会被添加到现有列表中
    return {"messages": response}

# 定义一个图
workflow = StateGraph(MessagesState)

# 定义我们将在其间循环的两个节点
workflow.add_node('agent', call_model)
workflow.add_node('action', tool_node)

# 添加一个条件边
workflow.add_conditional_edges(
    # 首先，我们定义起始节点。我们使用'agent'
    # 这意味着这些是在'agent'节点被调用后采取的边
    "agent",
    # 接下来，我们传入将确定下个调用哪个节点的函数
    should_continue,
    # 接下来，我们传入中路径映射-这条边可能去往的所有可能节点
    ['action', END]
)

# 现在我们从'tools'到'agent'添加一个普通边
workflow.add_edge("action", "agent")

# 最后，我们编译它
# 这将它编译成一个LangChain Runnable
# 意味着你可以像使用任何其他runnable一样使用它
# 设置检查点为内存形式，注意没有设置store
app = workflow.compile(checkpointer=memory)

调用图

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "2"}}
input_message = HumanMessage(content="hi, 我是tomie")
for event in app.stream({"messages": {input_message}}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()
    
input_message = HumanMessage(content="我叫什么名字")
for event in app.stream({"messages": {input_message}}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

In [ ]:
# 拉取MongoDB镜像
docker pull docker

# 启动MongoDB容器
docker run -d -p 27017:27017 --name mongodb mongo

# 查看MongoDB容器状态
docker ps

In [ ]:
## pip install -U pymongo langgraph-checkpoint-mongodb

In [3]:
import pymongo

# 创建MongoDB客户端连接
client = pymongo.MongoClient("mongodb://localhost:27017/")

# 测试连接
try:
    client.admin.command("ping")
    print("MongoDB连接成功")
except Exception as e:
    print(f"MongoDB连接失败: {e}")

MongoDB连接成功


创建一个最简单的智能体


In [ ]:
from typing import Literal
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent

@tool
def get_weather(city: Literal["北京", "深圳"]):
    """用来返回天气信息的工具函数"""
    if city == "北京":
        return "北京天气晴朗 大约22度 湿度30%"
    elif city == "深圳":
        return "深圳天气多云 大约28度 湿度60%"
    else:
        raise AssertionError("Unknown city")

tools = [get_weather]    
model = ChatDeepSeek(/*...*/)

连接MongoDB进行查询

In [ ]:
from langgraph.checkpoint.mongodb import MongoDBSaver

MONGODB_URI = "localhost:27017"

with MongoDBSaver.from_conn_string(MONGODB_URI) as checkpointer:
    graph = create_agent(model=model, tools=tools, checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "1"}}
    response = graph.invoke({"messages": [{"human", '北京今天的天气如何？'}]}, config=config)

优化记忆
***
- 消息过滤: 对旧消息进行类似删除或编辑的操作，目的是为了防止撑爆上下文
- 消息总结：对旧消息进行总结，目的一样是为了防止记忆内容过长
- 注意对记忆的管理一项关于召回率和精度的平衡艺术